<a href="https://colab.research.google.com/github/SophiaSama/agents/blob/main/Cross_Encoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install sentence-transformers -q

In [11]:
from google.colab import userdata
from huggingface_hub import login
import os

HF_TOKEN = userdata.get('HF_TOKEN')
if HF_TOKEN:
    # Log in to Hugging Face Hub using the retrieved token
    login(token=HF_TOKEN, add_to_git_credential=False)
    os.environ["HF_TOKEN"] = HF_TOKEN
else:
    print("Warning: HF_TOKEN not found in Colab Secrets.")

HfHubHTTPError: Invalid user token. The token from Google Colab vault is invalid. Please update it from the UI.

In [12]:
import os
from sentence_transformers import CrossEncoder

class LlmService:
    def __init__(self, model):
        # Initialize the local CrossEncoder model
        print(f"Loading CrossEncoder model '{model}' to local environment...")
        self.model = CrossEncoder(model)

    def rerank_documents(self, query, documents, top_n=5):
        # Prepare pairs of (query, document) for the CrossEncoder
        pairs = [[query, doc] for doc in documents]

        # Compute similarity scores
        scores = self.model.predict(pairs)

        # Combine documents with their index and scores
        ranked_results = []
        for idx, score in enumerate(scores):
            ranked_results.append({
                'index': idx,
                'score': float(score)
            })

        # Sort results based on score in descending order
        ranked_results.sort(key=lambda x: x['score'], reverse=True)
        return ranked_results[:top_n]

# Instantiate the service locally with BAAI/bge-reranker-large
# This model will run on Colab's CPU or GPU locally (no API keys or external endpoints needed!)
model_name = "BAAI/bge-reranker-large"
service = LlmService(model=model_name)

# Quick Demo test
test_query = "machine learning models in production"
test_docs = [
    "Running and deploying predictive systems in cloud environments.",
    "A delicious recipe for making chocolate chip cookies.",
    "Serving PyTorch and TensorFlow models to production clusters.",
    "History of industrial revolutions in Europe."
]

print(f"\nReranking with model: {model_name}...\n")
results = service.rerank_documents(query=test_query, documents=test_docs)

if results:
    for rank, item in enumerate(results):
        idx = item['index']
        score = item['score']
        print(f"Rank {rank+1}: Score {score:.4f} -> {test_docs[idx]}")

Loading CrossEncoder model 'BAAI/bge-reranker-large' to local environment...


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]


Reranking with model: BAAI/bge-reranker-large...

Rank 1: Score 0.3900 -> Serving PyTorch and TensorFlow models to production clusters.
Rank 2: Score 0.0002 -> Running and deploying predictive systems in cloud environments.
Rank 3: Score 0.0001 -> A delicious recipe for making chocolate chip cookies.
Rank 4: Score 0.0001 -> History of industrial revolutions in Europe.
